In [1]:
import pandas as pd
df = pd.read_csv('../data/raw/diabetic_data.csv')

print(df.shape)
print(df.dtypes)
print(df.isnull().sum())
print((df == '?').sum())   # '?' isn't NaN by default, check separately

(101766, 50)
encounter_id                int64
patient_nbr                 int64
race                          str
gender                        str
age                           str
weight                        str
admission_type_id           int64
discharge_disposition_id    int64
admission_source_id         int64
time_in_hospital            int64
payer_code                    str
medical_specialty             str
num_lab_procedures          int64
num_procedures              int64
num_medications             int64
number_outpatient           int64
number_emergency            int64
number_inpatient            int64
diag_1                        str
diag_2                        str
diag_3                        str
number_diagnoses            int64
max_glu_serum                 str
A1Cresult                     str
metformin                     str
repaglinide                   str
nateglinide                   str
chlorpropamide                str
glimepiride                   str
a

In [2]:
df = df.drop(columns=['weight', 'payer_code', 'encounter_id'])

In [3]:
df = df.replace('?', pd.NA)
df['race'] = df['race'].fillna('Unknown')
df['medical_specialty'] = df['medical_specialty'].fillna('Unknown')

In [4]:
before = len(df)
df = df.drop_duplicates(subset='patient_nbr', keep='first')
print(f"{before} -> {len(df)}")

101766 -> 71518


In [5]:
df = df[~df['discharge_disposition_id'].isin([11,13,14,19,20,21])]

In [6]:
df['readmitted_binary'] = (df['readmitted'] == '<30').astype(int)
print(df['readmitted_binary'].value_counts(normalize=True))

readmitted_binary
0    0.910294
1    0.089706
Name: proportion, dtype: float64


In [7]:
# ============================================================
# Step 1.7 — Bucket ICD-9 diagnosis codes into clinical chapters
# ============================================================

def map_icd9_to_group(code):
    if pd.isna(code):
        return 'Missing'
    code = str(code)

    # V/E codes = external causes / supplemental — bucket separately
    if code.startswith('V') or code.startswith('E'):
        return 'Other'

    try:
        code_num = float(code)
    except ValueError:
        return 'Other'

    # Diabetes (ICD-9 250.xx) checked first — most clinically relevant for this project
    if 250 <= code_num < 251:
        return 'Diabetes'
    elif 390 <= code_num < 460 or code_num == 785:
        return 'Circulatory'
    elif 460 <= code_num < 520 or code_num == 786:
        return 'Respiratory'
    elif 520 <= code_num < 580 or code_num == 787:
        return 'Digestive'
    elif 580 <= code_num < 630 or code_num == 788:
        return 'Genitourinary'
    elif 800 <= code_num < 1000:
        return 'Injury'
    elif 710 <= code_num < 740:
        return 'Musculoskeletal'
    elif 140 <= code_num < 240:
        return 'Neoplasm'
    else:
        return 'Other'

for col in ['diag_1', 'diag_2', 'diag_3']:
    df[f'{col}_group'] = df[col].apply(map_icd9_to_group)

print(df['diag_1_group'].value_counts())

diag_1_group
Circulatory        21384
Other              12122
Respiratory         9486
Digestive           6487
Diabetes            5748
Injury              4694
Musculoskeletal     4064
Genitourinary       3440
Neoplasm            2538
Missing               10
Name: count, dtype: int64


In [8]:
# ============================================================
# Step 1.8 — Bucket age bands into ordinal numeric feature
# ============================================================

age_map = {
    '[0-10)': 0, '[10-20)': 1, '[20-30)': 2, '[30-40)': 3, '[40-50)': 4,
    '[50-60)': 5, '[60-70)': 6, '[70-80)': 7, '[80-90)': 8, '[90-100)': 9
}
df['age_ordinal'] = df['age'].map(age_map)

print(df[['age', 'age_ordinal']].drop_duplicates().sort_values('age_ordinal'))

print(df['age_ordinal'].isnull().sum(), "nulls (should be 0)")

        age  age_ordinal
0    [0-10)            0
1   [10-20)            1
2   [20-30)            2
3   [30-40)            3
4   [40-50)            4
5   [50-60)            5
6   [60-70)            6
7   [70-80)            7
8   [80-90)            8
9  [90-100)            9
0 nulls (should be 0)


In [9]:
import os

# Ensure the directory exists
os.makedirs("data/processed", exist_ok=True)

for col in ['diag_1', 'diag_2', 'diag_3']:
    if col in df.columns:
        df[col] = df[col].fillna("Missing")

for col in ['max_glu_serum', 'A1Cresult']:
    if col in df.columns:
        df[col] = df[col].fillna("None")

df.to_csv('../data/processed/cleaned_data.csv', index=False)

print(df.shape)
print(df.columns.tolist())

(69973, 52)
['patient_nbr', 'race', 'gender', 'age', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted', 'readmitted_binary', 'diag_1_group', 'diag_2_group', 'diag_3_group', 'age_ordinal']


In [10]:
import pandas as pd
df = pd.read_csv("../data/processed/cleaned_data.csv")
print(df.isnull().sum().sum())  # should print 0


123753
